#PHASE 2 - EDA

In [ ]:
#Imports & Loads

import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
warnings.filterwarnings('ignore')



In [ ]:
# Setup & Data Loading
df = pd.read_csv('../data/processed/wine_combined.csv')

# parameters list
params = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
          'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
          'density', 'pH', 'sulphates', 'alcohol']

# Shared colour mapping 
WINE_COLORS = {'Red': 'crimson', 'White': 'palegoldenrod'}

print(f"Dataset loaded: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("✅ Ready")

In [ ]:
# Wine Quality Score Distribution

fig = px.histogram(df, x='quality', color='wine_type',
                   nbins=7,
                   title='Quality Score Distribution — Red vs White Wine',
                   labels={'quality': 'Quality Score', 'wine_type': 'Wine Type'},
                   color_discrete_map=WINE_COLORS,
                   opacity=0.8)

fig.update_layout(
    xaxis=dict(tickmode='linear', dtick=1),
    yaxis_title='Number of Samples',
    bargap=0.1,
    paper_bgcolor="#E8E8E8",   
    # plot_bgcolor='white', 
)
fig.show()

In [ ]:
# QC Pass/Fail Sunburst Chart

status_counts = df.groupby(['wine_type', 'qc_status']).size().reset_index(name='count')

fig = px.sunburst(status_counts,
                  path=['wine_type', 'qc_status'],
                  values='count',
                  title='QC Pass/Fail by Wine Type',
                  color='wine_type',
                  color_discrete_map=WINE_COLORS)

fig.update_traces(root_color='#E8E8E8')
# fig.update_layout(
#     paper_bgcolor='#E8E8E8',
# )

fig.show()

In [ ]:
# Box Plots of All Parameters (in loading cell) by Wine Type

for param in params:
    fig = px.box(df, x='wine_type', y=param,
                 color='wine_type',
                 title=f'{param.title()} — Red vs White Wine',
                 color_discrete_map=WINE_COLORS,
                 points=False) # no outlier dots needed

    fig.update_layout(
        paper_bgcolor='#E8E8E8',
        showlegend=False,
        yaxis_title=param.title(),
        xaxis_title='Wine Type',
        height=400
    )
    fig.show()

In [ ]:
# Violin Plots — (A view of parameter distributions by Quality Tier

for param in params:
    fig = px.violin(df, x='quality_tier', y=param,
                    color='wine_type',
                    box=True,
                    points=False,
                    category_orders={'quality_tier': ['Low', 'Medium', 'High']},
                    title=f'{param.title()} by Quality Tier',
                    color_discrete_map=WINE_COLORS,
                    labels={'quality_tier': 'Quality Tier', param: param.title()})

    fig.update_layout(
        paper_bgcolor="#E8E8E8",
        legend_title='Wine Type'
    )
    fig.show()

In [ ]:
# Correlation Heatmaps — Red & White Wine

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, wtype, cmap in zip(axes, ['Red', 'White'], ['Reds', 'YlOrBr']):
    corr = df[df['wine_type'] == wtype][params + ['quality']].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap=cmap, ax=ax,
                annot_kws={'size': 8}, linewidths=0.5)
    ax.set_title(f'{wtype} Wine — Correlation Matrix', fontsize=14, pad=12)

plt.suptitle('Physicochemical Parameter Correlations', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../assets/heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SO₂ Binding Ratio by Quality Tier. How is this ratio distributed across the tiers of wine quality?

fig = px.box(df, x='quality_tier', y='SO2_binding_ratio',
             color='wine_type',
             category_orders={'quality_tier': ['Low', 'Medium', 'High']},
             title='SO₂ Binding Ratio by Quality Tier',
             labels={'SO2_binding_ratio': 'Free SO₂ / Total SO₂',
                     'quality_tier': 'Quality Tier',
                     'wine_type': 'Wine Type'},
             color_discrete_map=WINE_COLORS,
             points=False)

fig.add_annotation(text="Higher ratio = more active SO₂ preservation",
                   xref="paper", yref="paper",
                   x=0.5, y=1.08, showarrow=False,
                   font=dict(size=11, color="grey"))

fig.update_layout(paper_bgcolor='#E8E8E8', legend_title='Wine Type')
fig.show()

In [ ]:
# START STUDYING YOUR CODE FROM HERE!!!

In [ ]:
# what wine characteristics influence Quality - Correlation Bar Chart


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, wtype in zip(axes, ['Red', 'White']):
    corr = df[df['wine_type'] == wtype][params + ['quality']].corr()['quality'].drop('quality').sort_values()

    # Color bars by direction — positive or negative correlation
    bar_colors = ['crimson' if x < 0 else 'palegoldenrod' for x in corr.values]

    ax.barh(corr.index, corr.values, color=bar_colors, edgecolor='white')
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_title(f'{wtype} Wine — Quality Drivers', fontsize=13)
    ax.set_xlabel('Correlation with Quality Score')
    ax.set_xlim(-0.6, 0.6)

    # Annotate each bar with its value
    for i, (val, namez) in enumerate(zip(corr.values, corr.index)):
        ax.text(val + (0.02 if val >= 0 else -0.02), i, f'{val:.2f}',
                va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.suptitle('What Drives Wine Quality? — Correlation Analysis', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../assets/quality_drivers.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chemical Profile of Low vs High Quality Wines
# How do the chemical properties differ between bad wines and good wines?

top_params = ['volatile acidity', 'alcohol', 'sulphates',
              'citric acid', 'pH', 'chlorides']

for wtype in ['Red', 'White']:
    subset = df[(df['wine_type'] == wtype) & (df['quality_tier'].isin(['Low', 'High']))]
    means  = subset.groupby('quality_tier')[top_params].mean().reset_index()
    melted = means.melt(id_vars='quality_tier', var_name='Parameter', value_name='Mean Value')

    fig = px.bar(melted, x='Parameter', y='Mean Value',
                 color='quality_tier',
                 barmode='group',
                 title=f'{wtype} Wine — Low vs High Quality Chemical Profile',
                 labels={'quality_tier': 'Quality Tier'},
                 color_discrete_map={'Low': 'tomato', 'High': 'seagreen'},
                 text_auto='.2f')

    fig.update_layout(
        paper_bgcolor='#E8E8E8',
        # plot_bgcolor='white',
        legend_title='Quality Tier',
        xaxis_tickangle=-30
    )
    fig.show()

In [ ]:
# Quick QC overview - Are Specifications Widely According to OIV / EU Standards

spec_limits = {
    'volatile acidity':    {'LSL': 0.08,  'USL': 1.2},   # OIV / US Federal
    'pH':                  {'LSL': 2.9,   'USL': 4.0},   # UC Davis
    'sulphates':           {'LSL': 0.25,  'USL': 1.5},   # Practical range
    'alcohol':             {'LSL': 8.5,   'USL': 15.0},  # EU Reg 1308/2013
    'free sulfur dioxide': {'LSL': 10.0,  'USL': 60.0},  # OIV Annex C
    'chlorides':           {'LSL': 0.005, 'USL': 0.20},  # Practical range
}

def classify_status(mean, lsl, usl):
    margin = 0.1 * (usl - lsl)
    if mean < lsl or mean > usl:             
        return '🔴 Out of Spec'
    elif mean < lsl + margin or mean > usl - margin: 
        return '🟡 Marginal'
    else:                                    
        return '🟢 In Spec'

rows = []
for wtype in ['Red', 'White']:
    subset = df[df['wine_type'] == wtype]
    for param, spec in spec_limits.items():
        rows.append({
            'Wine Type': wtype,
            'Parameter': param,
            'Mean': round(subset[param].mean(), 3),
            'Std Dev': round(subset[param].std(), 3),
            'LSL': spec['LSL'],
            'USL': spec['USL'],
            'Status': classify_status(subset[param].mean(), spec['LSL'], spec['USL'])
        })

status_df = pd.DataFrame(rows)

# Colour Formatting
status_colors = ['#d4edda' if s == '🟢 In Spec'
                 else '#fff3cd' if s == '🟡 Marginal'
                 else '#f8d7da' for s in status_df['Status']]

# Creating the Table
fig = go.Figure(data=[go.Table(
    columnwidth=[80, 150, 80, 80, 80, 80, 120],
    header=dict(values=list(status_df.columns),
                fill_color='#404040',
                font=dict(color='white', size=12),
                align='center', height=35),
    cells=dict(values=[status_df[col] for col in status_df.columns],
               fill_color=[['#f9f9f9'] * len(status_df)] * 6 + [status_colors],
               font=dict(size=11),
               align='center', height=30)
)])

fig.update_layout(
    title='Parameter Status Summary — QC Health Check (OIV/EU Standards)',
    paper_bgcolor='#E8E8E8',
    height=500
)
fig.show()

OPTIONAL ADDITIONAL VIZ.

In [65]:
# Radar Chart — Chemical Fingerprint Red vs White -optional

# Calculate normalised means per wine type (for fair comparison)
radar_df   = df.groupby('wine_type')[params].mean()
radar_norm = (radar_df - radar_df.min()) / (radar_df.max() - radar_df.min())

fig = go.Figure()

for wtype, color in WINE_COLORS.items():
    values = radar_norm.loc[wtype].tolist()
    values += values[:1]  # close the polygon

    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=params + [params[0]], #close the polygon
        fill='toself',
        name=wtype,
        line_color=color,
        fillcolor=color,
        opacity=0.4
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Chemical Fingerprint — Red vs White Wine',
    paper_bgcolor='#E8E8E8',
    legend_title='Wine Type'
)
fig.show()